In [ ]:
### COLAB SETUP

%cd /content
!rm -rf /content/proto-tsrl
!git clone https://github.com/haiyan-wang/proto-tsrl.git /content/proto-tsrl
%cd /content/proto-tsrl

from google.colab import drive, runtime
drive.mount('/content/drive')

import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)

In [ ]:
!pip install pacmap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix

import pacmap

from src.utils.training_utils import _pairwise_cos_sim
from src.utils.sampling_utils import TimeSeriesDataset
from src.experiments.ppg.ppg_model import PPGModel
from src.experiments.ppg.ppg_data_utils import *

In [ ]:
### SETTINGS

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# device
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)

# logging
SAVE_DIR = "/content/drive/MyDrive/Duke/Senior Year/Thesis/experiments/ppg"

# architecture
REPR_DIMS = [300]
MODELS = {}
for dim in REPR_DIMS:
    model_list = {}

    ckpt_dir = f"{SAVE_DIR}/checkpoints/dim{dim}"
    for ckpt_file in os.listdir(ckpt_dir):
        epoch = ckpt_file[10:-3]
        ckpt = torch.load(f"{ckpt_dir}/{ckpt_file}", map_location = "cpu")
        MODEL = PPGModel(representation_dimension = dim)
        MODEL.load_state_dict(ckpt)
        MODEL = MODEL.to(DEVICE)
        if torch.cuda.device_count() > 1:
            MODEL = nn.DataParallel(MODEL)
        elif not isinstance(MODEL, nn.DataParallel):
            MODEL = nn.DataParallel(MODEL)
        MODEL.eval()

        model_list[epoch] = MODEL

    MODELS[dim] = model_list

In [ ]:
### LOAD DATA

with np.load("/content/drive/MyDrive/Duke/Senior Year/Thesis/autoregressive_data/autoregressive.npz", allow_pickle = True) as data:
    series = list(data["series"])
y = np.loadtxt("/content/drive/MyDrive/Duke/Senior Year/Thesis/autoregressive_data/autoregressive_labels.csv", delimiter = ',')

# shuffle
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(series))
series = [series[i] for i in idx]
y = y[idx]

# split
split = int(0.95 * len(series))
train_series = series[:split]
train_labels = y[:split]
test_series = series[split:]
test_labels = y[split:]

ds_train, ds_test = TimeSeriesDataset(train_series), TimeSeriesDataset(test_series)

dl_train = DataLoader(
    ds_train,
    batch_size = 128,
    shuffle = True,
    num_workers = 4,
    pin_memory = True
)

dl_test = DataLoader(
    ds_test,
    batch_size = len(test_series),
    shuffle = False,
    num_workers = 4,
    pin_memory = True,
)

In [ ]:
### GENERATE REPRESENTATIONS

with torch.inference_mode():

    for dim in REPR_DIMS:
        model = MODELS[dim]['epoch40']
        repr_train, repr_test = model(X_train_tensor), model(X_test_tensor)
        repr_train, repr_test = repr_train.cpu().numpy(), repr_test.cpu().numpy()

        np.savetxt(f'{SAVE_DIR}/test_reprs/prototsrl{dim}-repr-train.csv', repr_train, delimiter = ',')
        np.savetxt(f'{SAVE_DIR}/test_reprs/prototsrl{dim}-repr-test.csv', repr_test, delimiter = ',')